In [4]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors, inchi
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
from rdkit.DataStructs import cDataStructs


# 1. Compare via Tanimoto similairties

In [5]:
df = pd.read_csv("/Users/hannah/code/rotation1/COVID_moonshot_submissions/covid_submissions_all_info.csv")

/var/folders/5n/b_7_p7h90vsbwf_xsbbwx4wc0000gn/T/ipykernel_68199/235806823.py:1: DtypeWarning: Columns (0: CID (old format), 1: creator, 2: rationale, 3: fragments, 4: Structure ID, 5: Fragalysis Link, 6: Enamine - REAL Space, 7: Enamine - Extended REAL Space, 8: Enamine - SCR, 9: Enamine - BB, 10: In Molport or Mcule, 11: In eMolecules, 12: Covalent Fragment, 13: covalent_warhead, 14: Acrylamide, 15: Acrylamide Adduct, 16: Chloroacetamide, 17: Chloroacetamide Adduct, 18: Vinylsulfonamide, 19: Vinylsulfonamide Adduct, 20: Nitrile, 21: Nitrile Adduct, 22: r_curve_IC50_x, 23: r_max_inhibition_reading_x, 24: r_min_inhibition_reading_x, 25: r_hill_slope_x, 26: r_R2_x, 27: r_concentration_uM_x, 28: r_inhibition_list_x, 29: f_curve_IC50_x, 30: f_max_inhibition_reading_x, 31: f_min_inhibition_reading_x, 32: f_hill_slope_x, 33: f_R2_x, 34: f_concentration_uM_x, 35: f_inhibition_list_x, 36: structure_ID_x, 37: structure_LINK_x, 38: creator.1, 39: rationale.1, 40: covalent_warhead.1, 41: r_curve

In [6]:
df_rIC50 = df[df["r_avg_IC50"].notna()] #filter out nas for rapid fire IC50 values
df_fIC50 = df[df["f_avg_IC50"].notna()] #filter out nas for fluoresence assay IC50 values

print(len(df_rIC50), len(df_fIC50)) #length matches the csv

877 2260


In [7]:
#extract their SMILES as a readable list
df_rIC50['mol'] = df_rIC50['SMILES'].apply(lambda x: Chem.MolFromSmiles(x) if Chem.MolFromSmiles is not None else None)
df_fIC50['mol'] = df_fIC50['SMILES'].apply(lambda x: Chem.MolFromSmiles(x) if Chem.MolFromSmiles is not None else None)

print(len(df_rIC50['mol']), len(df_fIC50['mol']))

877 2260


In [8]:
#get smiles from parsed fragalysis file - SARS-CoV-2 focused one
df_frag = pd.read_csv("/Users/hannah/code/rotation1/rotation1_code/Mpro_analysis/fragalysis_parsed_full_1319.csv")
df_frag['mol'] = df_frag["lig_smiles"].apply(lambda x: Chem.MolFromSmiles(x) if Chem.MolFromSmiles is not None else None)

print(len(df_frag['mol']))

1319


In [9]:
def mol_to_fps(mol_list):
    return [rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048) for mol in mol_list]

df_frag['fps'] = mol_to_fps(df_frag['mol'])
df_rIC50['fps'] = mol_to_fps(df_rIC50['mol'])
df_fIC50['fps'] = mol_to_fps(df_fIC50['mol'])

rIC50_frag_matrix = np.zeros((len(df_rIC50['fps']), len(df_frag['fps'])))
fIC50_frag_matrix = np.zeros((len(df_fIC50['fps']), len(df_frag['fps'])))


for i, fp_rIC50 in enumerate(df_rIC50['fps']):
    similarity = cDataStructs.BulkTanimotoSimilarity(fp_rIC50, df_frag['fps']) #compare one fp from fps_rIC50 to all fingerprints in fps_frag
    #faster than nested loop
    rIC50_frag_matrix[i] = similarity

for i, fp_rIC50 in enumerate(df_fIC50['fps']):
    similarity = cDataStructs.BulkTanimotoSimilarity(fp_rIC50, df_frag['fps']) #compare one fp from fps_rIC50 to all fingerprints in fps_frag
    #faster than nested loop
    fIC50_frag_matrix[i] = similarity

In [10]:
rIC50_matches = np.argwhere(rIC50_frag_matrix == 1)
fIC50_matches = np.argwhere(fIC50_frag_matrix == 1)
print(f"Number of exact matches (rIC50) using Tanimato similarity: {len(rIC50_matches)}")
print(f"Number of exact matches (fIC50) using Tanimato similarity: {len(fIC50_matches)}")


Number of exact matches (rIC50) using Tanimato similarity: 359
Number of exact matches (fIC50) using Tanimato similarity: 926


In [11]:
# Create empty columns
df_frag['rIC50_affinity_Tanimoto'] = np.nan
df_frag['fIC50_affinity_Tanimoto'] = np.nan

# Fill in affinity values using match indices
for i, j in rIC50_matches:
    df_frag.iloc[j, df_frag.columns.get_loc('rIC50_affinity_Tanimoto')] = df_rIC50.iloc[i]['r_avg_IC50']

for i, j in fIC50_matches:
    df_frag.iloc[j, df_frag.columns.get_loc('fIC50_affinity_Tanimoto')] = df_fIC50.iloc[i]['f_avg_IC50']

# Summary
print(f"rIC50 matched: {df_frag['rIC50_affinity_Tanimoto'].notna().sum()} / {len(df_frag)}")
print(f"fIC50 matched: {df_frag['fIC50_affinity_Tanimoto'].notna().sum()} / {len(df_frag)}")

rIC50 matched: 215 / 1319
fIC50 matched: 495 / 1319


# 2. Using InChiKey
more unique than SMILES (simplied molecular input line entry system) and less guessing work than canonical smiles

In [12]:
df_rIC50['InChIKey_cal'] = df_rIC50['mol'].apply(inchi.MolToInchiKey)
df_frag['InChIKey_cal'] = df_frag['mol'].apply(inchi.MolToInchiKey)
df_fIC50['InChIKey_cal'] = df_fIC50['mol'].apply(inchi.MolToInchiKey)

print(len(df_rIC50['InChIKey_cal']), len(df_frag['InChIKey_cal']), len(df_fIC50['InChIKey_cal']))

877 1319 2260


Calculating the InChIKey for the COVID Moonshot again, named InChIKey_cal, so i can match things up to be 100% clear 

In [13]:
match = dict(zip(df_rIC50['InChIKey'], df_rIC50['InChIKey_cal']))

df_rIC50['match'] = df_rIC50['InChIKey_cal'].map(match)
matched = df_rIC50['match'].notna().sum()
print(f"Matched {matched} / {len(df_rIC50)} molecules") #all the same so can use InChIKey

Matched 877 / 877 molecules


In [14]:
match = dict(zip(df_fIC50['InChIKey'], df_fIC50['InChIKey_cal']))

df_fIC50['match'] = df_fIC50['InChIKey_cal'].map(match)
matched = df_fIC50['match'].notna().sum()
print(f"Matched {matched} / {len(df_fIC50)} molecules") #all the same so can use InChIKey

Matched 2260 / 2260 molecules


In [15]:
rIC50_vs_frag_InChIKey = dict(zip(df_rIC50['InChIKey'], df_frag["InChIKey_cal"]))
df_frag['rIC50_affinity_InChIKey'] = df_frag['InChIKey_cal'].map(rIC50_vs_frag_InChIKey)

matched_rIC50_frag = df_frag[df_frag['rIC50_affinity_InChIKey'].notna()]
print(f"Matched: {len(matched_rIC50_frag)} / {len(df_rIC50)}")

Matched: 140 / 877


In [17]:
fIC50_vs_frag_InChIKey = dict(zip(df_fIC50['InChIKey'], df_fIC50["f_avg_IC50"]))
df_frag['fIC50_affinity_InChIKey'] = df_frag['InChIKey_cal'].map(fIC50_vs_frag_InChIKey)

matched_fIC50_frag = df_frag[df_frag['fIC50_affinity_InChIKey'].notna()]
print(f"Matched: {len(matched_fIC50_frag)} / {len(df_fIC50)}")

Matched: 300 / 2260


just say drop-out dont have time to really go detail into assessing why it dropped out

now going to match to see how many mol have both binding affinity data